<a href="https://colab.research.google.com/github/MuBasHarAkBar/darts-cifar10-nas/blob/main/DARTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!unzip -q /content/darts-master.zip -d /content/

In [3]:
!ls /content/

darts-master  darts-master.zip


In [4]:
%cd /content/darts-master/cnn

/content/darts-master/cnn


In [5]:
!pwd

/content/darts-master/cnn


In [6]:
!ls

architect.py  model_search.py	test.py		   train_search.py
genotypes.py  operations.py	train_imagenet.py  utils.py
model.py      test_imagenet.py	train.py	   visualize.py


In [7]:
!nvidia-smi

Mon Oct  5 12:40:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

PyTorch version: 2.11.0+cu130
Torchvision version: 0.26.0+cu130
CUDA available: True
CUDA version: 13.0
GPU: Tesla T4
GPU count: 1


In [9]:
!mkdir -p /content/cifar10

In [10]:
!ls /content

cifar10  darts-master  darts-master.zip


In [11]:
!find /content -maxdepth 3 -name "train_search.py"

/content/darts-master/cnn/train_search.py
/content/darts-master/rnn/train_search.py


In [12]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.CIFAR10(
    root="/content/cifar10",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="/content/cifar10",
    train=False,
    download=True,
    transform=transform
)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

100%|██████████| 170M/170M [12:37<00:00, 225kB/s]


Training images: 50000
Testing images: 10000


In [13]:
!cp -r /content/darts-master/cnn /content/cnn_backup

In [14]:
%cd /content/darts-master/cnn

/content/darts-master/cnn


In [15]:
!ls

architect.py  model_search.py	test.py		   train_search.py
genotypes.py  operations.py	train_imagenet.py  utils.py
model.py      test_imagenet.py	train.py	   visualize.py


In [16]:
!cp -r /content/darts-master/cnn /content/cnn_original_backup

In [17]:
!ls /content/cnn_original_backup

architect.py  model_search.py	test.py		   train_search.py
genotypes.py  operations.py	train_imagenet.py  utils.py
model.py      test_imagenet.py	train.py	   visualize.py


In [18]:
!sed -n '1,260p' train_search.py

import os
import sys
import time
import glob
import numpy as np
import torch
import utils
import logging
import argparse
import torch.nn as nn
import torch.utils
import torch.nn.functional as F
import torchvision.datasets as dset
import torch.backends.cudnn as cudnn

from torch.autograd import Variable
from model_search import Network
from architect import Architect


parser = argparse.ArgumentParser("cifar")
parser.add_argument('--data', type=str, default='../data', help='location of the data corpus')
parser.add_argument('--batch_size', type=int, default=64, help='batch size')
parser.add_argument('--learning_rate', type=float, default=0.025, help='init learning rate')
parser.add_argument('--learning_rate_min', type=float, default=0.001, help='min learning rate')
parser.add_argument('--momentum', type=float, default=0.9, help='momentum')
parser.add_argument('--weight_decay', type=float, default=3e-4, help='weight decay')
parser.add_argument('--report_freq', type=float, default=50, help

In [19]:
!grep -n "Variable\|\.data\[\|volatile\|cuda\|tensor(" train_search.py

16:from torch.autograd import Variable
61:  if not torch.cuda.is_available():
66:  torch.cuda.set_device(args.gpu)
70:  torch.cuda.manual_seed(args.seed)
75:  criterion = criterion.cuda()
77:  model = model.cuda()
139:    input = Variable(input, requires_grad=False).cuda()
140:    target = Variable(target, requires_grad=False).cuda(async=True)
144:    input_search = Variable(input_search, requires_grad=False).cuda()
145:    target_search = Variable(target_search, requires_grad=False).cuda(async=True)
158:    objs.update(loss.data[0], n)
159:    top1.update(prec1.data[0], n)
160:    top5.update(prec5.data[0], n)
175:    input = Variable(input, volatile=True).cuda()
176:    target = Variable(target, volatile=True).cuda(async=True)
183:    objs.update(loss.data[0], n)
184:    top1.update(prec1.data[0], n)
185:    top5.update(prec5.data[0], n)


In [20]:
!grep -n "Variable\|\.data\[\|volatile\|cuda\|tensor(" architect.py

4:from torch.autograd import Variable
57:        v.grad = Variable(g.data)
74:    return model_new.cuda()


In [21]:
!grep -n "Variable\|\.data\[\|volatile\|cuda\|tensor(" model_search.py

5:from torch.autograd import Variable
98:    model_new = Network(self._C, self._num_classes, self._layers, self._criterion).cuda()
123:    self.alphas_normal = Variable(1e-3*torch.randn(k, num_ops).cuda(), requires_grad=True)
124:    self.alphas_reduce = Variable(1e-3*torch.randn(k, num_ops).cuda(), requires_grad=True)


In [22]:
!sed -n '1,320p' architect.py

import torch
import numpy as np
import torch.nn as nn
from torch.autograd import Variable


def _concat(xs):
  return torch.cat([x.view(-1) for x in xs])


class Architect(object):

  def __init__(self, model, args):
    self.network_momentum = args.momentum
    self.network_weight_decay = args.weight_decay
    self.model = model
    self.optimizer = torch.optim.Adam(self.model.arch_parameters(),
        lr=args.arch_learning_rate, betas=(0.5, 0.999), weight_decay=args.arch_weight_decay)

  def _compute_unrolled_model(self, input, target, eta, network_optimizer):
    loss = self.model._loss(input, target)
    theta = _concat(self.model.parameters()).data
    try:
      moment = _concat(network_optimizer.state[v]['momentum_buffer'] for v in self.model.parameters()).mul_(self.network_momentum)
    except:
      moment = torch.zeros_like(theta)
    dtheta = _concat(torch.autograd.grad(loss, self.model.parameters())).data + self.network_weight_decay*theta
    unrolled_model = self._constru

In [23]:
!sed -n '1,420p' model_search.py

import torch
import torch.nn as nn
import torch.nn.functional as F
from operations import *
from torch.autograd import Variable
from genotypes import PRIMITIVES
from genotypes import Genotype


class MixedOp(nn.Module):

  def __init__(self, C, stride):
    super(MixedOp, self).__init__()
    self._ops = nn.ModuleList()
    for primitive in PRIMITIVES:
      op = OPS[primitive](C, stride, False)
      if 'pool' in primitive:
        op = nn.Sequential(op, nn.BatchNorm2d(C, affine=False))
      self._ops.append(op)

  def forward(self, x, weights):
    return sum(w * op(x) for w, op in zip(weights, self._ops))


class Cell(nn.Module):

  def __init__(self, steps, multiplier, C_prev_prev, C_prev, C, reduction, reduction_prev):
    super(Cell, self).__init__()
    self.reduction = reduction

    if reduction_prev:
      self.preprocess0 = FactorizedReduce(C_prev_prev, C, affine=False)
    else:
      self.preprocess0 = ReLUConvBN(C_prev_prev, C, 1, 1, 0, affine=False)
    self.preprocess1

In [24]:
!sed -n '1,420p' utils.py

import os
import numpy as np
import torch
import shutil
import torchvision.transforms as transforms
from torch.autograd import Variable


class AvgrageMeter(object):

  def __init__(self):
    self.reset()

  def reset(self):
    self.avg = 0
    self.sum = 0
    self.cnt = 0

  def update(self, val, n=1):
    self.sum += val * n
    self.cnt += n
    self.avg = self.sum / self.cnt


def accuracy(output, target, topk=(1,)):
  maxk = max(topk)
  batch_size = target.size(0)

  _, pred = output.topk(maxk, 1, True, True)
  pred = pred.t()
  correct = pred.eq(target.view(1, -1).expand_as(pred))

  res = []
  for k in topk:
    correct_k = correct[:k].view(-1).float().sum(0)
    res.append(correct_k.mul_(100.0/batch_size))
  return res


class Cutout(object):
    def __init__(self, length):
        self.length = length

    def __call__(self, img):
        h, w = img.size(1), img.size(2)
        mask = np.ones((h, w), np.float32)
        y = np.random.randint(h)
        x = np.random.randint

In [25]:
import os

os.makedirs("/content/darts_modern", exist_ok=True)

print("Folder created:", os.path.exists("/content/darts_modern"))

Folder created: True


In [26]:
%%writefile /content/darts_modern/architect.py

import torch
import numpy as np
import torch.nn as nn


def _concat(xs):
    return torch.cat([x.view(-1) for x in xs])


class Architect(object):

    def __init__(self, model, args):
        self.network_momentum = args.momentum
        self.network_weight_decay = args.weight_decay
        self.model = model

        self.optimizer = torch.optim.Adam(
            self.model.arch_parameters(),
            lr=args.arch_learning_rate,
            betas=(0.5, 0.999),
            weight_decay=args.arch_weight_decay
        )

    def _compute_unrolled_model(
        self,
        input,
        target,
        eta,
        network_optimizer
    ):
        loss = self.model._loss(input, target)

        theta = _concat(self.model.parameters()).detach()

        try:
            moment = _concat(
                network_optimizer.state[v]['momentum_buffer']
                for v in self.model.parameters()
            ).mul(self.network_momentum)
        except KeyError:
            moment = torch.zeros_like(theta)

        gradients = torch.autograd.grad(
            loss,
            self.model.parameters(),
            create_graph=False
        )

        dtheta = _concat(gradients).detach()
        dtheta = dtheta + self.network_weight_decay * theta

        unrolled_theta = theta - eta * (moment + dtheta)

        unrolled_model = self._construct_model_from_theta(
            unrolled_theta
        )

        return unrolled_model

    def step(
        self,
        input_train,
        target_train,
        input_valid,
        target_valid,
        eta,
        network_optimizer,
        unrolled
    ):
        self.optimizer.zero_grad()

        if unrolled:
            self._backward_step_unrolled(
                input_train,
                target_train,
                input_valid,
                target_valid,
                eta,
                network_optimizer
            )
        else:
            self._backward_step(
                input_valid,
                target_valid
            )

        self.optimizer.step()

    def _backward_step(self, input_valid, target_valid):
        loss = self.model._loss(
            input_valid,
            target_valid
        )

        loss.backward()

    def _backward_step_unrolled(
        self,
        input_train,
        target_train,
        input_valid,
        target_valid,
        eta,
        network_optimizer
    ):

        unrolled_model = self._compute_unrolled_model(
            input_train,
            target_train,
            eta,
            network_optimizer
        )

        unrolled_loss = unrolled_model._loss(
            input_valid,
            target_valid
        )

        unrolled_loss.backward()

        dalpha = [
            v.grad for v in unrolled_model.arch_parameters()
        ]

        vector = [
            v.grad for v in unrolled_model.parameters()
        ]

        implicit_grads = self._hessian_vector_product(
            vector,
            input_train,
            target_train
        )

        for g, ig in zip(dalpha, implicit_grads):
            g.data.sub_(eta * ig.data)

        for v, g in zip(
            self.model.arch_parameters(),
            dalpha
        ):
            if v.grad is None:
                v.grad = g.detach().clone()
            else:
                v.grad.data.copy_(g.data)

    def _construct_model_from_theta(self, theta):

        model_new = self.model.new()

        model_dict = self.model.state_dict()

        params = {}
        offset = 0

        for k, v in self.model.named_parameters():

            v_length = np.prod(v.size())

            params[k] = theta[
                offset:offset + v_length
            ].view(v.size())

            offset += v_length

        assert offset == len(theta)

        model_dict.update(params)

        model_new.load_state_dict(
            model_dict,
            strict=False
        )

        return model_new.cuda()

    def _hessian_vector_product(
        self,
        vector,
        input,
        target,
        r=1e-2
    ):

        vector_norm = _concat(vector).norm()

        if vector_norm.item() == 0:
            return [
                torch.zeros_like(alpha)
                for alpha in self.model.arch_parameters()
            ]

        R = r / vector_norm

        for p, v in zip(
            self.model.parameters(),
            vector
        ):
            p.data.add_(R * v)

        loss = self.model._loss(
            input,
            target
        )

        grads_p = torch.autograd.grad(
            loss,
            self.model.arch_parameters(),
            allow_unused=True
        )

        for p, v in zip(
            self.model.parameters(),
            vector
        ):
            p.data.sub_((2 * R) * v)

        loss = self.model._loss(
            input,
            target
        )

        grads_n = torch.autograd.grad(
            loss,
            self.model.arch_parameters(),
            allow_unused=True
        )

        for p, v in zip(
            self.model.parameters(),
            vector
        ):
            p.data.add_(R * v)

        result = []

        for x, y in zip(grads_p, grads_n):

            if x is None:
                x = torch.zeros_like(y)

            if y is None:
                y = torch.zeros_like(x)

            result.append(
                (x - y) / (2 * R)
            )

        return result

Writing /content/darts_modern/architect.py


In [27]:
%%writefile /content/darts_modern/model_search.py

import torch
import torch.nn as nn
import torch.nn.functional as F

from operations import *
from genotypes import PRIMITIVES
from genotypes import Genotype


class MixedOp(nn.Module):

    def __init__(self, C, stride):
        super(MixedOp, self).__init__()

        self._ops = nn.ModuleList()

        for primitive in PRIMITIVES:

            op = OPS[primitive](
                C,
                stride,
                False
            )

            if 'pool' in primitive:
                op = nn.Sequential(
                    op,
                    nn.BatchNorm2d(
                        C,
                        affine=False
                    )
                )

            self._ops.append(op)

    def forward(self, x, weights):

        return sum(
            w * op(x)
            for w, op in zip(
                weights,
                self._ops
            )
        )


class Cell(nn.Module):

    def __init__(
        self,
        steps,
        multiplier,
        C_prev_prev,
        C_prev,
        C,
        reduction,
        reduction_prev
    ):

        super(Cell, self).__init__()

        self.reduction = reduction

        if reduction_prev:

            self.preprocess0 = FactorizedReduce(
                C_prev_prev,
                C,
                affine=False
            )

        else:

            self.preprocess0 = ReLUConvBN(
                C_prev_prev,
                C,
                1,
                1,
                0,
                affine=False
            )

        self.preprocess1 = ReLUConvBN(
            C_prev,
            C,
            1,
            1,
            0,
            affine=False
        )

        self._steps = steps
        self._multiplier = multiplier

        self._ops = nn.ModuleList()

        for i in range(self._steps):

            for j in range(2 + i):

                stride = (
                    2
                    if reduction and j < 2
                    else 1
                )

                op = MixedOp(
                    C,
                    stride
                )

                self._ops.append(op)

    def forward(
        self,
        s0,
        s1,
        weights
    ):

        s0 = self.preprocess0(s0)
        s1 = self.preprocess1(s1)

        states = [
            s0,
            s1
        ]

        offset = 0

        for i in range(self._steps):

            s = sum(
                self._ops[offset + j](
                    h,
                    weights[offset + j]
                )
                for j, h in enumerate(states)
            )

            offset += len(states)

            states.append(s)

        return torch.cat(
            states[-self._multiplier:],
            dim=1
        )


class Network(nn.Module):

    def __init__(
        self,
        C,
        num_classes,
        layers,
        criterion,
        steps=4,
        multiplier=4,
        stem_multiplier=3
    ):

        super(Network, self).__init__()

        self._C = C
        self._num_classes = num_classes
        self._layers = layers
        self._criterion = criterion
        self._steps = steps
        self._multiplier = multiplier

        C_curr = stem_multiplier * C

        self.stem = nn.Sequential(
            nn.Conv2d(
                3,
                C_curr,
                3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(C_curr)
        )

        C_prev_prev = C_curr
        C_prev = C_curr
        C_curr = C

        self.cells = nn.ModuleList()

        reduction_prev = False

        for i in range(layers):

            if i in [
                layers // 3,
                2 * layers // 3
            ]:

                C_curr *= 2
                reduction = True

            else:

                reduction = False

            cell = Cell(
                steps,
                multiplier,
                C_prev_prev,
                C_prev,
                C_curr,
                reduction,
                reduction_prev
            )

            reduction_prev = reduction

            self.cells.append(cell)

            C_prev_prev = C_prev
            C_prev = multiplier * C_curr

        self.global_pooling = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Linear(
            C_prev,
            num_classes
        )

        self._initialize_alphas()

    def new(self):

        model_new = Network(
            self._C,
            self._num_classes,
            self._layers,
            self._criterion,
            self._steps,
            self._multiplier
        ).cuda()

        for x, y in zip(
            model_new.arch_parameters(),
            self.arch_parameters()
        ):

            x.data.copy_(y.data)

        return model_new

    def forward(self, input):

        s0 = self.stem(input)
        s1 = self.stem(input)

        for cell in self.cells:

            if cell.reduction:

                weights = F.softmax(
                    self.alphas_reduce,
                    dim=-1
                )

            else:

                weights = F.softmax(
                    self.alphas_normal,
                    dim=-1
                )

            s0, s1 = (
                s1,
                cell(
                    s0,
                    s1,
                    weights
                )
            )

        out = self.global_pooling(s1)

        logits = self.classifier(
            out.view(out.size(0), -1)
        )

        return logits

    def _loss(self, input, target):

        logits = self(input)

        return self._criterion(
            logits,
            target
        )

    def _initialize_alphas(self):

        k = sum(
            1
            for i in range(self._steps)
            for n in range(2 + i)
        )

        num_ops = len(PRIMITIVES)

        self.alphas_normal = nn.Parameter(
            1e-3 * torch.randn(
                k,
                num_ops,
                device='cuda'
            )
        )

        self.alphas_reduce = nn.Parameter(
            1e-3 * torch.randn(
                k,
                num_ops,
                device='cuda'
            )
        )

        self._arch_parameters = [
            self.alphas_normal,
            self.alphas_reduce
        ]

    def arch_parameters(self):

        return self._arch_parameters

    def genotype(self):

        def _parse(weights):

            gene = []

            n = 2
            start = 0

            for i in range(self._steps):

                end = start + n

                W = weights[
                    start:end
                ].copy()

                edges = sorted(
                    range(i + 2),
                    key=lambda x:
                        -max(
                            W[x][k]
                            for k in range(len(W[x]))
                            if k != PRIMITIVES.index('none')
                        )
                )[:2]

                for j in edges:

                    k_best = None

                    for k in range(len(W[j])):

                        if k != PRIMITIVES.index('none'):

                            if (
                                k_best is None
                                or W[j][k] > W[j][k_best]
                            ):

                                k_best = k

                    gene.append(
                        (
                            PRIMITIVES[k_best],
                            j
                        )
                    )

                start = end
                n += 1

            return gene

        gene_normal = _parse(
            F.softmax(
                self.alphas_normal,
                dim=-1
            ).detach().cpu().numpy()
        )

        gene_reduce = _parse(
            F.softmax(
                self.alphas_reduce,
                dim=-1
            ).detach().cpu().numpy()
        )

        concat = range(
            2 + self._steps - self._multiplier,
            self._steps + 2
        )

        genotype = Genotype(
            normal=gene_normal,
            normal_concat=concat,
            reduce=gene_reduce,
            reduce_concat=concat
        )

        return genotype

Writing /content/darts_modern/model_search.py


In [28]:
%%writefile /content/darts_modern/utils.py

import os
import numpy as np
import torch
import shutil
import torchvision.transforms as transforms


class AvgrageMeter(object):

    def __init__(self):
        self.reset()

    def reset(self):
        self.avg = 0
        self.sum = 0
        self.cnt = 0

    def update(self, val, n=1):
        self.sum += val * n
        self.cnt += n
        self.avg = self.sum / self.cnt


def accuracy(output, target, topk=(1,)):

    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(
        maxk,
        1,
        True,
        True
    )

    pred = pred.t()

    correct = pred.eq(
        target.view(1, -1).expand_as(pred)
    )

    res = []

    for k in topk:

        correct_k = (
            correct[:k]
            .reshape(-1)
            .float()
            .sum(0)
        )

        res.append(
            correct_k.mul(
                100.0 / batch_size
            )
        )

    return res


class Cutout(object):

    def __init__(self, length):
        self.length = length

    def __call__(self, img):

        h, w = img.size(1), img.size(2)

        mask = np.ones(
            (h, w),
            np.float32
        )

        y = np.random.randint(h)
        x = np.random.randint(w)

        y1 = np.clip(
            y - self.length // 2,
            0,
            h
        )

        y2 = np.clip(
            y + self.length // 2,
            0,
            h
        )

        x1 = np.clip(
            x - self.length // 2,
            0,
            w
        )

        x2 = np.clip(
            x + self.length // 2,
            0,
            w
        )

        mask[
            y1:y2,
            x1:x2
        ] = 0.

        mask = torch.from_numpy(mask)

        mask = mask.expand_as(img)

        img = img * mask

        return img


def _data_transforms_cifar10(args):

    CIFAR_MEAN = [
        0.49139968,
        0.48215827,
        0.44653124
    ]

    CIFAR_STD = [
        0.24703233,
        0.24348505,
        0.26158768
    ]

    train_transform = transforms.Compose([
        transforms.RandomCrop(
            32,
            padding=4
        ),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            CIFAR_MEAN,
            CIFAR_STD
        ),
    ])

    if args.cutout:

        train_transform.transforms.append(
            Cutout(args.cutout_length)
        )

    valid_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            CIFAR_MEAN,
            CIFAR_STD
        ),
    ])

    return train_transform, valid_transform


def count_parameters_in_MB(model):

    return np.sum(
        np.prod(v.size())
        for name, v in model.named_parameters()
        if "auxiliary" not in name
    ) / 1e6


def save_checkpoint(
    state,
    is_best,
    save
):

    filename = os.path.join(
        save,
        'checkpoint.pth.tar'
    )

    torch.save(
        state,
        filename
    )

    if is_best:

        best_filename = os.path.join(
            save,
            'model_best.pth.tar'
        )

        shutil.copyfile(
            filename,
            best_filename
        )


def save(model, model_path):

    torch.save(
        model.state_dict(),
        model_path
    )


def load(model, model_path):

    model.load_state_dict(
        torch.load(
            model_path,
            map_location='cuda'
        )
    )


def drop_path(x, drop_prob):

    if drop_prob > 0.:

        keep_prob = 1. - drop_prob

        mask = torch.empty(
            x.size(0),
            1,
            1,
            1,
            device=x.device
        ).bernoulli_(keep_prob)

        x = x / keep_prob

        x = x * mask

    return x


def create_exp_dir(
    path,
    scripts_to_save=None
):

    if not os.path.exists(path):

        os.mkdir(path)

    print(
        'Experiment dir : {}'.format(path)
    )

    if scripts_to_save is not None:

        os.mkdir(
            os.path.join(
                path,
                'scripts'
            )
        )

        for script in scripts_to_save:

            dst_file = os.path.join(
                path,
                'scripts',
                os.path.basename(script)
            )

            shutil.copyfile(
                script,
                dst_file
            )

Writing /content/darts_modern/utils.py


In [29]:
%%writefile /content/darts_modern/train_search.py

import os
import sys
import time
import glob
import numpy as np
import torch
import utils
import logging
import argparse
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as dset
import torch.backends.cudnn as cudnn

from model_search import Network
from architect import Architect


parser = argparse.ArgumentParser("cifar")

parser.add_argument(
    '--data',
    type=str,
    default='/content/cifar10',
    help='location of the data corpus'
)

parser.add_argument(
    '--batch_size',
    type=int,
    default=64
)

parser.add_argument(
    '--learning_rate',
    type=float,
    default=0.025
)

parser.add_argument(
    '--learning_rate_min',
    type=float,
    default=0.001
)

parser.add_argument(
    '--momentum',
    type=float,
    default=0.9
)

parser.add_argument(
    '--weight_decay',
    type=float,
    default=3e-4
)

parser.add_argument(
    '--report_freq',
    type=float,
    default=50
)

parser.add_argument(
    '--gpu',
    type=int,
    default=0
)

parser.add_argument(
    '--epochs',
    type=int,
    default=2
)

parser.add_argument(
    '--init_channels',
    type=int,
    default=16
)

parser.add_argument(
    '--layers',
    type=int,
    default=8
)

parser.add_argument(
    '--model_path',
    type=str,
    default='saved_models'
)

parser.add_argument(
    '--cutout',
    action='store_true',
    default=False
)

parser.add_argument(
    '--cutout_length',
    type=int,
    default=16
)

parser.add_argument(
    '--drop_path_prob',
    type=float,
    default=0.3
)

parser.add_argument(
    '--save',
    type=str,
    default='EXP'
)

parser.add_argument(
    '--seed',
    type=int,
    default=2
)

parser.add_argument(
    '--grad_clip',
    type=float,
    default=5
)

parser.add_argument(
    '--train_portion',
    type=float,
    default=0.5
)

parser.add_argument(
    '--unrolled',
    action='store_true',
    default=False
)

parser.add_argument(
    '--arch_learning_rate',
    type=float,
    default=3e-4
)

parser.add_argument(
    '--arch_weight_decay',
    type=float,
    default=1e-3
)

args = parser.parse_args()

args.save = 'search-{}-{}'.format(
    args.save,
    time.strftime("%Y%m%d-%H%M%S")
)

utils.create_exp_dir(
    args.save,
    scripts_to_save=glob.glob('*.py')
)

log_format = '%(asctime)s %(message)s'

logging.basicConfig(
    stream=sys.stdout,
    level=logging.INFO,
    format=log_format,
    datefmt='%m/%d %I:%M:%S %p'
)

fh = logging.FileHandler(
    os.path.join(
        args.save,
        'log.txt'
    )
)

fh.setFormatter(
    logging.Formatter(log_format)
)

logging.getLogger().addHandler(fh)


CIFAR_CLASSES = 10


def main():

    if not torch.cuda.is_available():

        logging.info(
            'no gpu device available'
        )

        sys.exit(1)

    np.random.seed(args.seed)

    torch.manual_seed(args.seed)
    torch.cuda.manual_seed(args.seed)

    cudnn.benchmark = True
    cudnn.enabled = True

    device = torch.device(
        'cuda:{}'.format(args.gpu)
    )

    logging.info(
        'gpu device = %s',
        torch.cuda.get_device_name(args.gpu)
    )

    logging.info(
        'args = %s',
        args
    )

    criterion = nn.CrossEntropyLoss()
    criterion = criterion.to(device)

    model = Network(
        args.init_channels,
        CIFAR_CLASSES,
        args.layers,
        criterion
    )

    model = model.to(device)

    logging.info(
        'param size = %fMB',
        utils.count_parameters_in_MB(model)
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=args.learning_rate,
        momentum=args.momentum,
        weight_decay=args.weight_decay
    )

    train_transform, valid_transform = (
        utils._data_transforms_cifar10(args)
    )

    train_data = dset.CIFAR10(
        root=args.data,
        train=True,
        download=True,
        transform=train_transform
    )

    num_train = len(train_data)

    indices = list(range(num_train))

    split = int(
        np.floor(
            args.train_portion * num_train
        )
    )

    train_queue = torch.utils.data.DataLoader(
        train_data,
        batch_size=args.batch_size,
        sampler=torch.utils.data.SubsetRandomSampler(
            indices[:split]
        ),
        pin_memory=True,
        num_workers=2
    )

    valid_queue = torch.utils.data.DataLoader(
        train_data,
        batch_size=args.batch_size,
        sampler=torch.utils.data.SubsetRandomSampler(
            indices[split:num_train]
        ),
        pin_memory=True,
        num_workers=2
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=args.epochs,
        eta_min=args.learning_rate_min
    )

    architect = Architect(
        model,
        args
    )

    for epoch in range(args.epochs):

        lr = scheduler.get_last_lr()[0]

        logging.info(
            'epoch %d lr %e',
            epoch,
            lr
        )

        genotype = model.genotype()

        logging.info(
            'genotype = %s',
            genotype
        )

        print(
            F.softmax(
                model.alphas_normal,
                dim=-1
            ).detach().cpu()
        )

        print(
            F.softmax(
                model.alphas_reduce,
                dim=-1
            ).detach().cpu()
        )

        train_acc, train_obj = train(
            train_queue,
            valid_queue,
            model,
            architect,
            criterion,
            optimizer,
            lr
        )

        logging.info(
            'train_acc %f',
            train_acc
        )

        valid_acc, valid_obj = infer(
            valid_queue,
            model,
            criterion
        )

        logging.info(
            'valid_acc %f',
            valid_acc
        )

        utils.save(
            model,
            os.path.join(
                args.save,
                'weights.pt'
            )
        )

        scheduler.step()


def train(
    train_queue,
    valid_queue,
    model,
    architect,
    criterion,
    optimizer,
    lr
):

    objs = utils.AvgrageMeter()
    top1 = utils.AvgrageMeter()
    top5 = utils.AvgrageMeter()

    valid_iter = iter(valid_queue)

    for step, (input, target) in enumerate(
        train_queue
    ):

        model.train()

        n = input.size(0)

        input = input.cuda(
            non_blocking=True
        )

        target = target.cuda(
            non_blocking=True
        )

        try:

            input_search, target_search = (
                next(valid_iter)
            )

        except StopIteration:

            valid_iter = iter(valid_queue)

            input_search, target_search = (
                next(valid_iter)
            )

        input_search = input_search.cuda(
            non_blocking=True
        )

        target_search = target_search.cuda(
            non_blocking=True
        )

        architect.step(
            input,
            target,
            input_search,
            target_search,
            lr,
            optimizer,
            unrolled=args.unrolled
        )

        optimizer.zero_grad()

        logits = model(input)

        loss = criterion(
            logits,
            target
        )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            args.grad_clip
        )

        optimizer.step()

        prec1, prec5 = utils.accuracy(
            logits,
            target,
            topk=(1, 5)
        )

        objs.update(
            loss.item(),
            n
        )

        top1.update(
            prec1.item(),
            n
        )

        top5.update(
            prec5.item(),
            n
        )

        if step % args.report_freq == 0:

            logging.info(
                'train %03d %e %f %f',
                step,
                objs.avg,
                top1.avg,
                top5.avg
            )

    return top1.avg, objs.avg


def infer(
    valid_queue,
    model,
    criterion
):

    objs = utils.AvgrageMeter()
    top1 = utils.AvgrageMeter()
    top5 = utils.AvgrageMeter()

    model.eval()

    with torch.no_grad():

        for step, (input, target) in enumerate(
            valid_queue
        ):

            input = input.cuda(
                non_blocking=True
            )

            target = target.cuda(
                non_blocking=True
            )

            logits = model(input)

            loss = criterion(
                logits,
                target
            )

            prec1, prec5 = utils.accuracy(
                logits,
                target,
                topk=(1, 5)
            )

            n = input.size(0)

            objs.update(
                loss.item(),
                n
            )

            top1.update(
                prec1.item(),
                n
            )

            top5.update(
                prec5.item(),
                n
            )

            if step % args.report_freq == 0:

                logging.info(
                    'valid %03d %e %f %f',
                    step,
                    objs.avg,
                    top1.avg,
                    top5.avg
                )

    return top1.avg, objs.avg


if __name__ == '__main__':
    main()

Writing /content/darts_modern/train_search.py


In [30]:
%%writefile /content/darts_modern/operations.py

import torch
import torch.nn as nn
import torch.nn.functional as F


OPS = {}


class ReLUConvBN(nn.Module):

    def __init__(self, C_in, C_out, kernel_size, stride, padding, affine=True):
        super().__init__()

        self.op = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(
                C_in,
                C_out,
                kernel_size,
                stride=stride,
                padding=padding,
                bias=False
            ),
            nn.BatchNorm2d(
                C_out,
                affine=affine
            )
        )

    def forward(self, x):
        return self.op(x)


class Identity(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x


class Zero(nn.Module):

    def __init__(self, stride):
        super().__init__()
        self.stride = stride

    def forward(self, x):

        if self.stride == 1:
            return x.mul(0.)

        return x[:, :, ::self.stride, ::self.stride].mul(0.)


class FactorizedReduce(nn.Module):

    def __init__(self, C_in, C_out, affine=True):
        super().__init__()

        assert C_out % 2 == 0

        self.relu = nn.ReLU(inplace=False)

        self.conv_1 = nn.Conv2d(
            C_in,
            C_out // 2,
            1,
            stride=2,
            padding=0,
            bias=False
        )

        self.conv_2 = nn.Conv2d(
            C_in,
            C_out // 2,
            1,
            stride=2,
            padding=0,
            bias=False
        )

        self.bn = nn.BatchNorm2d(
            C_out,
            affine=affine
        )

    def forward(self, x):

        x = self.relu(x)

        out = torch.cat(
            [
                self.conv_1(x),
                self.conv_2(x[:, :, 1:, 1:])
            ],
            dim=1
        )

        return self.bn(out)


class SepConv(nn.Module):

    def __init__(
        self,
        C_in,
        C_out,
        kernel_size,
        stride,
        padding,
        affine=True
    ):
        super().__init__()

        self.op = nn.Sequential(
            nn.ReLU(inplace=False),

            nn.Conv2d(
                C_in,
                C_in,
                kernel_size,
                stride=stride,
                padding=padding,
                groups=C_in,
                bias=False
            ),

            nn.Conv2d(
                C_in,
                C_in,
                kernel_size=1,
                padding=0,
                bias=False
            ),

            nn.BatchNorm2d(
                C_in,
                affine=affine
            ),

            nn.ReLU(inplace=False),

            nn.Conv2d(
                C_in,
                C_in,
                kernel_size,
                stride=1,
                padding=padding,
                groups=C_in,
                bias=False
            ),

            nn.Conv2d(
                C_in,
                C_out,
                kernel_size=1,
                padding=0,
                bias=False
            ),

            nn.BatchNorm2d(
                C_out,
                affine=affine
            )
        )

    def forward(self, x):
        return self.op(x)


class DilConv(nn.Module):

    def __init__(
        self,
        C_in,
        C_out,
        kernel_size,
        stride,
        padding,
        dilation,
        affine=True
    ):
        super().__init__()

        self.op = nn.Sequential(
            nn.ReLU(inplace=False),

            nn.Conv2d(
                C_in,
                C_in,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                dilation=dilation,
                groups=C_in,
                bias=False
            ),

            nn.Conv2d(
                C_in,
                C_out,
                kernel_size=1,
                padding=0,
                bias=False
            ),

            nn.BatchNorm2d(
                C_out,
                affine=affine
            )
        )

    def forward(self, x):
        return self.op(x)


class MaxPool3x3(nn.Module):

    def __init__(self, stride, padding=1):
        super().__init__()

        self.op = nn.MaxPool2d(
            3,
            stride=stride,
            padding=padding
        )

    def forward(self, x):
        return self.op(x)


class AvgPool3x3(nn.Module):

    def __init__(self, stride, padding=1):
        super().__init__()

        self.op = nn.AvgPool2d(
            3,
            stride=stride,
            padding=padding,
            count_include_pad=False
        )

    def forward(self, x):
        return self.op(x)


OPS = {
    'none': lambda C, stride, affine:
        Zero(stride),

    'avg_pool_3x3': lambda C, stride, affine:
        AvgPool3x3(stride),

    'max_pool_3x3': lambda C, stride, affine:
        MaxPool3x3(stride),

    'skip_connect': lambda C, stride, affine:
        Identity() if stride == 1 else FactorizedReduce(
            C,
            C,
            affine=affine
        ),

    'sep_conv_3x3': lambda C, stride, affine:
        SepConv(
            C,
            C,
            3,
            stride,
            1,
            affine=affine
        ),

    'sep_conv_5x5': lambda C, stride, affine:
        SepConv(
            C,
            C,
            5,
            stride,
            2,
            affine=affine
        ),

    'dil_conv_3x3': lambda C, stride, affine:
        DilConv(
            C,
            C,
            3,
            stride,
            2,
            2,
            affine=affine
        ),

    'dil_conv_5x5': lambda C, stride, affine:
        DilConv(
            C,
            C,
            5,
            stride,
            4,
            2,
            affine=affine
        ),
}

Writing /content/darts_modern/operations.py


In [31]:
%%writefile /content/darts_modern/genotypes.py

from collections import namedtuple


Genotype = namedtuple(
    'Genotype',
    'normal normal_concat reduce reduce_concat'
)


PRIMITIVES = [
    'none',
    'max_pool_3x3',
    'avg_pool_3x3',
    'skip_connect',
    'sep_conv_3x3',
    'sep_conv_5x5',
    'dil_conv_3x3',
    'dil_conv_5x5',
]

Writing /content/darts_modern/genotypes.py


In [32]:
%cd /content/darts_modern

/content/darts_modern


In [33]:
import torch
import torch.nn as nn

from model_search import Network

criterion = nn.CrossEntropyLoss().cuda()

model = Network(
    C=16,
    num_classes=10,
    layers=8,
    criterion=criterion
).cuda()

print("Model created successfully")
print("GPU:", torch.cuda.get_device_name(0))
print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)
print(
    "Architecture parameters:",
    [
        p.shape
        for p in model.arch_parameters()
    ]
)

Model created successfully
GPU: Tesla T4
Parameters: 1930842
Architecture parameters: [torch.Size([14, 8]), torch.Size([14, 8])]


In [34]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        [0.49139968, 0.48215827, 0.44653124],
        [0.24703233, 0.24348505, 0.26158768]
    )
])

dataset = torchvision.datasets.CIFAR10(
    root="/content/cifar10",
    train=True,
    download=False,
    transform=transform
)

loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2
)

images, labels = next(iter(loader))

images = images.cuda()
labels = labels.cuda()

logits = model(images)

print("Input:", images.shape)
print("Output:", logits.shape)

Input: torch.Size([8, 3, 32, 32])
Output: torch.Size([8, 10])


In [35]:
loss = criterion(logits, labels)

print("Loss:", loss.item())

loss.backward()

print(
    "Normal alpha gradient:",
    model.alphas_normal.grad is not None
)

print(
    "Reduction alpha gradient:",
    model.alphas_reduce.grad is not None
)

Loss: 2.140011787414551
Normal alpha gradient: True
Reduction alpha gradient: True


In [36]:
%cd /content/darts_modern

/content/darts_modern


In [37]:
%%writefile model_search.py

import torch
import torch.nn as nn
import torch.nn.functional as F

from operations import *
from genotypes import PRIMITIVES
from genotypes import Genotype


class MixedOp(nn.Module):

    def __init__(self, C, stride):
        super(MixedOp, self).__init__()

        self._ops = nn.ModuleList()

        for primitive in PRIMITIVES:
            op = OPS[primitive](C, stride, False)

            if 'pool' in primitive:
                op = nn.Sequential(
                    op,
                    nn.BatchNorm2d(C, affine=False)
                )

            self._ops.append(op)

    def forward(self, x, weights):
        return sum(
            w * op(x)
            for w, op in zip(weights, self._ops)
        )


class Cell(nn.Module):

    def __init__(
        self,
        steps,
        multiplier,
        C_prev_prev,
        C_prev,
        C,
        reduction,
        reduction_prev
    ):
        super(Cell, self).__init__()

        self.reduction = reduction

        if reduction_prev:
            self.preprocess0 = FactorizedReduce(
                C_prev_prev,
                C,
                affine=False
            )
        else:
            self.preprocess0 = ReLUConvBN(
                C_prev_prev,
                C,
                1,
                1,
                0,
                affine=False
            )

        self.preprocess1 = ReLUConvBN(
            C_prev,
            C,
            1,
            1,
            0,
            affine=False
        )

        self._steps = steps
        self._multiplier = multiplier

        self._ops = nn.ModuleList()

        for i in range(self._steps):

            for j in range(2 + i):

                stride = 2 if reduction and j < 2 else 1

                op = MixedOp(C, stride)

                self._ops.append(op)

    def forward(self, s0, s1, weights):

        s0 = self.preprocess0(s0)
        s1 = self.preprocess1(s1)

        states = [s0, s1]

        offset = 0

        for i in range(self._steps):

            s = sum(
                self._ops[offset + j](
                    h,
                    weights[offset + j]
                )
                for j, h in enumerate(states)
            )

            offset += len(states)

            states.append(s)

        return torch.cat(
            states[-self._multiplier:],
            dim=1
        )


class Network(nn.Module):

    def __init__(
        self,
        C,
        num_classes,
        layers,
        criterion,
        steps=4,
        multiplier=4,
        stem_multiplier=3
    ):
        super(Network, self).__init__()

        self._C = C
        self._num_classes = num_classes
        self._layers = layers
        self._criterion = criterion

        self._steps = steps
        self._multiplier = multiplier

        C_curr = stem_multiplier * C

        self.stem = nn.Sequential(
            nn.Conv2d(
                3,
                C_curr,
                3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(C_curr)
        )

        C_prev_prev = C_curr
        C_prev = C_curr

        C_curr = C

        self.cells = nn.ModuleList()

        reduction_prev = False

        for i in range(layers):

            if i in [layers // 3, 2 * layers // 3]:

                C_curr *= 2
                reduction = True

            else:
                reduction = False

            cell = Cell(
                steps,
                multiplier,
                C_prev_prev,
                C_prev,
                C_curr,
                reduction,
                reduction_prev
            )

            reduction_prev = reduction

            self.cells.append(cell)

            C_prev_prev = C_prev
            C_prev = multiplier * C_curr

        self.global_pooling = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Linear(
            C_prev,
            num_classes
        )

        self._initialize_alphas()

    def new(self):

        model_new = Network(
            self._C,
            self._num_classes,
            self._layers,
            self._criterion,
            self._steps,
            self._multiplier
        ).cuda()

        model_new.alphas_normal = self.alphas_normal.detach().clone()
        model_new.alphas_normal.requires_grad = True

        model_new.alphas_reduce = self.alphas_reduce.detach().clone()
        model_new.alphas_reduce.requires_grad = True

        return model_new

    def forward(self, input):

        weights_normal = F.softmax(
            self.alphas_normal,
            dim=-1
        )

        weights_reduce = F.softmax(
            self.alphas_reduce,
            dim=-1
        )

        s0 = self.stem(input)
        s1 = self.stem(input)

        for i, cell in enumerate(self.cells):

            if cell.reduction:

                weights = weights_reduce

            else:

                weights = weights_normal

            s0, s1 = s1, cell(s0, s1, weights)

        out = self.global_pooling(s1)

        logits = self.classifier(
            out.view(out.size(0), -1)
        )

        return logits

    def _loss(self, input, target):

        logits = self(input)

        return self._criterion(
            logits,
            target
        )

    def arch_parameters(self):

        return [
            self.alphas_normal,
            self.alphas_reduce
        ]

    def genotype(self):

        def _parse(weights):

            gene = []

            n = 2

            start = 0

            for i in range(self._steps):

                end = start + n

                W = weights[start:end].copy()

                edges = sorted(
                    range(i + 2),
                    key=lambda x: -max(
                        W[x][k]
                        for k in range(len(W[x]))
                        if k != PRIMITIVES.index('none')
                    )
                )[:2]

                for j in edges:

                    k_best = None

                    for k in range(len(W[j])):

                        if k == PRIMITIVES.index('none'):
                            continue

                        if k_best is None or W[j][k] > W[j][k_best]:
                            k_best = k

                    gene.append(
                        (
                            PRIMITIVES[k_best],
                            j
                        )
                    )

                start = end
                n += 1

            return gene

        gene_normal = _parse(
            F.softmax(
                self.alphas_normal,
                dim=-1
            ).detach().cpu().numpy()
        )

        gene_reduce = _parse(
            F.softmax(
                self.alphas_reduce,
                dim=-1
            ).detach().cpu().numpy()
        )

        concat = range(
            2 + self._steps - self._multiplier,
            self._steps + 2
        )

        genotype = Genotype(
            normal=gene_normal,
            normal_concat=concat,
            reduce=gene_reduce,
            reduce_concat=concat
        )

        return genotype

    def _initialize_alphas(self):

        k = sum(
            1 for i in range(self._steps)
            for n in range(2 + i)
        )

        num_ops = len(PRIMITIVES)

        # IMPORTANT:
        # Keep architecture parameters separate from
        # normal network parameters, as in original DARTS.

        self.alphas_normal = (
            1e-3 *
            torch.randn(
                k,
                num_ops,
                device='cuda'
            )
        )

        self.alphas_normal.requires_grad_(True)

        self.alphas_reduce = (
            1e-3 *
            torch.randn(
                k,
                num_ops,
                device='cuda'
            )
        )

        self.alphas_reduce.requires_grad_(True)

Overwriting model_search.py


In [38]:
import torch
import torch.nn as nn

from model_search import Network

criterion = nn.CrossEntropyLoss().cuda()

model = Network(
    C=16,
    num_classes=10,
    layers=8,
    criterion=criterion
).cuda()

print("Model created successfully")
print("GPU:", torch.cuda.get_device_name(0))

print(
    "Network parameters:",
    sum(p.numel() for p in model.parameters())
)

print(
    "Architecture parameters:",
    [p.shape for p in model.arch_parameters()]
)

print(
    "Architecture parameters included in model.parameters():",
    any(
        p is model.alphas_normal or
        p is model.alphas_reduce
        for p in model.parameters()
    )
)

Model created successfully
GPU: Tesla T4
Network parameters: 1930842
Architecture parameters: [torch.Size([14, 8]), torch.Size([14, 8])]
Architecture parameters included in model.parameters(): True


In [39]:
images, labels = next(iter(loader))

images = images.cuda()
labels = labels.cuda()

logits = model(images)

loss = criterion(logits, labels)

loss.backward()

print("Input:", images.shape)
print("Output:", logits.shape)
print("Loss:", loss.item())

print(
    "Normal alpha gradient:",
    model.alphas_normal.grad is not None
)

print(
    "Reduction alpha gradient:",
    model.alphas_reduce.grad is not None
)

Input: torch.Size([8, 3, 32, 32])
Output: torch.Size([8, 10])
Loss: 2.3756442070007324
Normal alpha gradient: True
Reduction alpha gradient: True


In [40]:
%cd /content/darts_modern

/content/darts_modern


In [41]:
def _initialize_alphas(self):

    k = sum(
        1 for i in range(self._steps)
        for n in range(2 + i)
    )

    num_ops = len(PRIMITIVES)

    # Architecture parameters are intentionally NOT nn.Parameter.
    # This keeps them separate from model.parameters(),
    # matching the original DARTS implementation.

    self.alphas_normal = (
        1e-3 *
        torch.randn(
            k,
            num_ops,
            device='cuda',
            requires_grad=True
        )
    )

    self.alphas_reduce = (
        1e-3 *
        torch.randn(
            k,
            num_ops,
            device='cuda',
            requires_grad=True
        )
    )

In [42]:
%cd /content/darts_modern



/content/darts_modern


In [43]:
import torch
import torch.nn as nn

from model_search import Network

criterion = nn.CrossEntropyLoss().cuda()

model = Network(
    C=16,
    num_classes=10,
    layers=8,
    criterion=criterion
).cuda()

In [44]:
print("Architecture parameters:")
print(model.alphas_normal.shape)
print(model.alphas_reduce.shape)

print("\nRegistered model parameters containing 'alpha':")

for name, p in model.named_parameters():
    if "alpha" in name.lower():
        print(name)

Architecture parameters:
torch.Size([14, 8])
torch.Size([14, 8])

Registered model parameters containing 'alpha':
alphas_normal
alphas_reduce


In [45]:
arch_ids = {
    id(model.alphas_normal),
    id(model.alphas_reduce)
}

included = any(
    id(p) in arch_ids
    for p in model.parameters()
)

print(
    "Architecture parameters included in model.parameters():",
    included
)

Architecture parameters included in model.parameters(): True


In [46]:
%cd /content/darts_modern

/content/darts_modern


In [47]:
!grep -n "epochs\|batch_size\|data=" train_search.py | head -20

30:    '--batch_size',
72:    '--epochs',
272:        batch_size=args.batch_size,
282:        batch_size=args.batch_size,
292:        T_max=args.epochs,
301:    for epoch in range(args.epochs):


In [48]:
!python train_search.py --epochs 1

Experiment dir : search-EXP-20261005-125806
10/05 12:58:06 PM gpu device = Tesla T4
10/05 12:58:06 PM args = Namespace(data='/content/cifar10', batch_size=64, learning_rate=0.025, learning_rate_min=0.001, momentum=0.9, weight_decay=0.0003, report_freq=50, gpu=0, epochs=1, init_channels=16, layers=8, model_path='saved_models', cutout=False, cutout_length=16, drop_path_prob=0.3, save='search-EXP-20261005-125806', seed=2, grad_clip=5, train_portion=0.5, unrolled=False, arch_learning_rate=0.0003, arch_weight_decay=0.001)
10/05 12:58:06 PM param size = 1.930618MB
10/05 12:58:07 PM epoch 0 lr 2.500000e-02
10/05 12:58:07 PM genotype = Genotype(normal=[('sep_conv_3x3', 1), ('avg_pool_3x3', 0), ('skip_connect', 2), ('dil_conv_5x5', 0), ('skip_connect', 3), ('dil_conv_3x3', 2), ('dil_conv_3x3', 3), ('skip_connect', 0)], normal_concat=range(2, 6), reduce=[('dil_conv_5x5', 0), ('skip_connect', 1), ('dil_conv_5x5', 2), ('avg_pool_3x3', 0), ('dil_conv_3x3', 2), ('dil_conv_3x3', 3), ('avg_pool_3x3', 

In [ ]:
%cd /content/darts_modern

!python train_search.py --epochs 50

/content/darts_modern
Experiment dir : search-EXP-20261005-131038
10/05 01:10:38 PM gpu device = Tesla T4
10/05 01:10:38 PM args = Namespace(data='/content/cifar10', batch_size=64, learning_rate=0.025, learning_rate_min=0.001, momentum=0.9, weight_decay=0.0003, report_freq=50, gpu=0, epochs=50, init_channels=16, layers=8, model_path='saved_models', cutout=False, cutout_length=16, drop_path_prob=0.3, save='search-EXP-20261005-131038', seed=2, grad_clip=5, train_portion=0.5, unrolled=False, arch_learning_rate=0.0003, arch_weight_decay=0.001)
10/05 01:10:39 PM param size = 1.930618MB
10/05 01:10:40 PM epoch 0 lr 2.500000e-02
10/05 01:10:40 PM genotype = Genotype(normal=[('sep_conv_3x3', 1), ('avg_pool_3x3', 0), ('skip_connect', 2), ('dil_conv_5x5', 0), ('skip_connect', 3), ('dil_conv_3x3', 2), ('dil_conv_3x3', 3), ('skip_connect', 0)], normal_concat=range(2, 6), reduce=[('dil_conv_5x5', 0), ('skip_connect', 1), ('dil_conv_5x5', 2), ('avg_pool_3x3', 0), ('dil_conv_3x3', 2), ('dil_conv_3x3'